# Keyword Final

상품명 매칭/DB 저장용 최종 확인 노트북입니다.

흐름:
- 원본 `name`은 절대 수정하지 않고 보존합니다.
- 클러스터링용 전처리는 `cluster_name_text`에 저장합니다.
- 붙어쓴 상품명까지 토큰화한 결과는 `tokenized_cluster_name_text`에 저장합니다.
- 단계별 클러스터링 결과는 `c1`, `c2`, ...에 저장합니다.
- DB 저장용 매칭 상품명 후보는 `cluster_product_name`입니다.

In [8]:
from pathlib import Path
import re

import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer


def resolve_analysis_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == "analysis":
        return cwd

    candidate = cwd / "code/backend/src/main/python/analysis"
    if candidate.exists():
        return candidate

    return Path(__file__).resolve().parent if "__file__" in globals() else cwd


ANALYSIS_DIR = resolve_analysis_dir()
PYTHON_DIR = ANALYSIS_DIR.parent
CRAWLING_RESULTS_DIR = PYTHON_DIR / "crawling" / "results"
TARGET_CSV_NAME = "통합조회_전체_no_filter_20260605_1142.csv"

csv_path = CRAWLING_RESULTS_DIR / TARGET_CSV_NAME
if not csv_path.exists():
    raise FileNotFoundError(f"지정한 no-filter 크롤링 CSV를 찾을 수 없습니다: {csv_path}")

csv_path

WindowsPath('C:/project/kdtproject/kdtproject/code/backend/src/main/python/crawling/results/통합조회_전체_no_filter_20260605_1142.csv')

In [9]:
# 여기에 크롤링 단계에서 제외하고 싶은 name 포함 키워드를 추가하세요.
DROP_NAME_KEYWORDS = [
    "삽니다",
    "구매합니다",
    "매입",
    "대여",
    "교환",
    "렌탈",
    "매입",
    "대여",
]


def normalize_search_text(value):
    text = str(value or "").lower()
    text = re.sub(r"(?<=[a-z0-9])plus\b", " 플러스", text)
    text = re.sub(r"\bplus\b|[+＋]", " 플러스 ", text)
    text = re.sub(r"\bpro\b", " 프로 ", text)
    text = re.sub(r"\bmax\b", " 맥스 ", text)
    text = re.sub(r"\bultra\b", " 울트라 ", text)
    text = re.sub(r"[^0-9a-z가-힣\s]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_search_text(value):
    return re.sub(r"\s+", "", normalize_search_text(value))


def build_drop_name_keywords(keywords):
    drop_keywords = []
    seen = set()

    for keyword in keywords:
        original = str(keyword or "").strip()
        normalized = normalize_search_text(original)
        compact = compact_search_text(original)
        key = (normalized, compact)

        if not normalized or key in seen:
            continue

        seen.add(key)
        drop_keywords.append(
            {
                "original": original,
                "normalized": normalized,
                "compact": compact,
            }
        )

    return drop_keywords


DROP_NAME_KEYWORD_MATCHERS = build_drop_name_keywords(DROP_NAME_KEYWORDS)
NORMALIZED_DROP_NAME_KEYWORDS = [keyword["normalized"] for keyword in DROP_NAME_KEYWORD_MATCHERS]


def matched_drop_keywords(name):
    raw_name = str(name or "").lower()
    normalized_name = normalize_search_text(name)
    compact_name = compact_search_text(name)

    matched_keywords = []
    for keyword in DROP_NAME_KEYWORD_MATCHERS:
        raw_keyword = keyword["original"].lower()
        normalized_keyword = keyword["normalized"]
        compact_keyword = keyword["compact"]

        if (
            raw_keyword in raw_name
            or normalized_keyword in normalized_name
            or compact_keyword in compact_name
        ):
            matched_keywords.append(keyword["original"])

    return matched_keywords

In [ ]:
df = pd.read_csv(csv_path, encoding="utf-8-sig")

working_df = df.copy()
working_df["price_numeric"] = pd.to_numeric(working_df["price"], errors="coerce")
working_df = working_df.dropna(subset=["keyword", "name", "price_numeric"])
working_df = working_df[working_df["price_numeric"] > 0].copy()
working_df["price_numeric"] = working_df["price_numeric"].astype(int)

working_df["matched_drop_keywords"] = working_df["name"].apply(matched_drop_keywords)
working_df["name_blacklist_drop"] = working_df["matched_drop_keywords"].apply(bool)
working_df["matched_drop_keywords_text"] = working_df["matched_drop_keywords"].apply(lambda values: " | ".join(values))

filtered_df = working_df[~working_df["name_blacklist_drop"]].copy()
dropped_filter_df = working_df[working_df["name_blacklist_drop"]].copy()

filter_overview_df = pd.DataFrame([
    {
        "source_file": csv_path.name,
        "valid_price_rows_before_filter": len(working_df),
        "rows_after_name_blacklist_filter": len(filtered_df),
        "removed_by_name_blacklist_filter": len(dropped_filter_df),
        "name_blacklist_keep_rate": len(filtered_df) / len(working_df) if len(working_df) else 0,
        "keyword_count_before_filter": working_df["keyword"].nunique(),
        "keyword_count_after_filter": filtered_df["keyword"].nunique(),
        "drop_keyword_count": len(NORMALIZED_DROP_NAME_KEYWORDS),
    }
])

filter_overview_df

In [ ]:
LOW_PRICE_MEDIAN_RATIO = 0.2

keyword_stats_df = (
    filtered_df.groupby("keyword")["price_numeric"]
    .agg(
        item_count="count",
        min_price="min",
        q1=lambda values: values.quantile(0.25),
        median_price="median",
        q3=lambda values: values.quantile(0.75),
        max_price="max",
        average_price="mean",
    )
    .reset_index()
)

keyword_stats_df["iqr"] = keyword_stats_df["q3"] - keyword_stats_df["q1"]
keyword_stats_df["iqr_lower_bound"] = (keyword_stats_df["q1"] - 1.5 * keyword_stats_df["iqr"]).clip(lower=0)
keyword_stats_df["median_ratio_lower_bound"] = keyword_stats_df["median_price"] * LOW_PRICE_MEDIAN_RATIO
keyword_stats_df["lower_bound"] = keyword_stats_df[["iqr_lower_bound", "median_ratio_lower_bound"]].max(axis=1)
keyword_stats_df["upper_bound"] = keyword_stats_df["q3"] + 1.5 * keyword_stats_df["iqr"]

outlier_df = filtered_df.merge(
    keyword_stats_df[[
        "keyword",
        "q1",
        "median_price",
        "q3",
        "iqr",
        "iqr_lower_bound",
        "median_ratio_lower_bound",
        "lower_bound",
        "upper_bound",
    ]],
    on="keyword",
    how="left",
)

outlier_df["outlier_type"] = ""
outlier_df.loc[outlier_df["price_numeric"] < outlier_df["lower_bound"], "outlier_type"] = "low"
outlier_df.loc[outlier_df["price_numeric"] > outlier_df["upper_bound"], "outlier_type"] = "high"
outlier_df = outlier_df[outlier_df["outlier_type"] != ""].copy()
outlier_df["price_to_median_ratio"] = outlier_df["price_numeric"] / outlier_df["median_price"]

outlier_keys = outlier_df[["keyword", "platform", "pid", "price_numeric"]].copy()
outlier_keys["is_outlier"] = True

clean_price_df = filtered_df.merge(
    outlier_keys[["keyword", "platform", "pid", "price_numeric", "is_outlier"]],
    on=["keyword", "platform", "pid", "price_numeric"],
    how="left",
)
clean_price_df["is_outlier"] = clean_price_df["is_outlier"].fillna(False).astype(bool)
clean_price_df = clean_price_df.loc[~clean_price_df["is_outlier"]].copy()

clean_keyword_price_summary_df = (
    clean_price_df.groupby("keyword")["price_numeric"]
    .agg(
        clean_item_count="count",
        clean_min_price="min",
        clean_max_price="max",
        clean_average_price="mean",
        clean_median_price="median",
    )
    .reset_index()
)

filtered_counts_df = filtered_df.groupby("keyword").size().reset_index(name="name_blacklist_filtered_item_count")
outlier_counts_df = outlier_df.groupby("keyword").size().reset_index(name="removed_price_outlier_count")

clean_keyword_price_summary_df = filtered_counts_df.merge(
    outlier_counts_df,
    on="keyword",
    how="left",
).merge(
    clean_keyword_price_summary_df,
    on="keyword",
    how="left",
)
clean_keyword_price_summary_df["removed_price_outlier_count"] = clean_keyword_price_summary_df["removed_price_outlier_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["clean_item_count"] = clean_keyword_price_summary_df["clean_item_count"].fillna(0).astype(int)
clean_keyword_price_summary_df["removed_price_outlier_rate"] = clean_keyword_price_summary_df["removed_price_outlier_count"] / clean_keyword_price_summary_df["name_blacklist_filtered_item_count"]

clean_keyword_price_summary_df = clean_keyword_price_summary_df[[
    "keyword",
    "name_blacklist_filtered_item_count",
    "removed_price_outlier_count",
    "removed_price_outlier_rate",
    "clean_item_count",
    "clean_min_price",
    "clean_max_price",
    "clean_average_price",
    "clean_median_price",
]].sort_values("keyword")

clean_keyword_price_summary_df

,keyword,name_blacklist_filtered_item_count,removed_price_outlier_count,removed_price_outlier_rate,clean_item_count,clean_min_price,clean_max_price,clean_average_price,clean_median_price
0,갤럭시,10259,2230,0.217370,8029,32000,680000,213084.665338,165000.0
1,아이폰,10044,3828,0.381123,6216,22000,580000,168906.216699,138650.0


In [ ]:
# 상품명 매칭에 방해되는 거래/상태/배송 문구입니다.
CLUSTER_TRADE_NOISE_PATTERN = re.compile(
    r"판매합니다|판매해요|판매중|판매완료|판매|"
    r"팝니다|팔아요|팔아봅니다|팔아여|"
    r"구매합니다|구해요|구합니다|삽니다|매입|"
    r"급처합니다|급처|급매|일괄판매|일괄|"
    r"택포|착불|배송|직거래|거래|네고|에눌|"
    r"미개봉|새상품|신품|중고|정품|풀박스|풀박|단품|"
    r"교환|대여|렌탈"
)

# 상태/외관/등급 표기는 모델 구분이 아니므로 제거합니다.
CLUSTER_CONDITION_NOISE_PATTERN = re.compile(
    r"무잔상|유잔상|잔상|무기스|기스|스크래치|"
    r"께끗|깨끗|깔끔|클린|깨끗한|깨끗한기기|"
    r"(?:aa|aaa|ss|s|a|b|c|d|e)급|"
    r"리퍼급|새상품급|중고급|민트급|미사용급|"
    r"정상공기기|정상해지|공기기|공기계|자급제|해지"
)

# 색상/지역명은 모델명 매칭에 불필요하므로 제거합니다.
CLUSTER_COLOR_NOISE_PATTERN = re.compile(
    r"블랙|화이트|실버|골드|그레이|그린|레드|블루|"
    r"핑크|퍼플|바이올렛|라벤더|민트|네이비|"
    r"옐로우|크림|베이지|코랄|오렌지|차콜|"
    r"티타늄|티타니움|아이스블루|딥퍼플|스카이블루|"
    r"메탈릭|코퍼|브론즈|로즈골드|스페이스그레이|"
    r"그래파이트|실버쉐도우|엠버|옐로우|바이올렛|"
    r"라이트그린|다크그레이|미드나잇|스타라이트|"
    r"퍼플|블루쉐도우|엠버옐로우|메탈릭코퍼"
)

CLUSTER_REGION_NOISE_PATTERN = re.compile(
    r"서울|부산|대구|인천|광주|대전|울산|세종|"
    r"경기|강원|충북|충남|전북|전남|경북|경남|제주|"
    r"수원|성남|고양|용인|부천|안산|안양|남양주|화성|평택|"
    r"의정부|시흥|파주|김포|광명|하남|군포|김해|창원|진주|"
    r"포항|제주|천안|청주|전주|목포|여수|순천|양산|구미|안동|"
    r"일산|분당|판교|강남|홍대|신촌|건대|잠실|"
    r"부산중고폰|인천중고폰|김해중고폰|양산중고폰"
)

CLUSTER_NUMERIC_TRACKING_CODE_PATTERN = re.compile(
    r"[\(\[\{]\s*\d{3,}\s*[\)\]\}]|"  # 예: (7818), [25773]
    r"^\s*\d{3,}\s+|"  # 예: 749 갤럭시와이드6
    r"\s+\d{3,}\s*$"  # 예: 갤럭시노트8블루 1925
)

# 토큰 단계에서도 제외할 잔여 토큰입니다. 무잔상 분해 시 생기는 '무', '잔상' 등을 막습니다.
TOKEN_STAGE_EXCLUDED_TOKENS = frozenset({
    "무", "잔상", "유", "기스", "무기스", "급", "a급", "b급", "s급", "ss급",
    "aa급", "aaa급", "리퍼급", "새상품급", "께끗", "깨끗", "깔끔",
})


def remove_cluster_numeric_tracking_codes(value):
    text = str(value or "")
    text = CLUSTER_NUMERIC_TRACKING_CODE_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def cluster_normalized_name(value):
    text = remove_cluster_numeric_tracking_codes(value)
    text = normalize_search_text(text)
    text = CLUSTER_TRADE_NOISE_PATTERN.sub(" ", text)
    text = CLUSTER_CONDITION_NOISE_PATTERN.sub(" ", text)
    text = CLUSTER_COLOR_NOISE_PATTERN.sub(" ", text)
    text = CLUSTER_REGION_NOISE_PATTERN.sub(" ", text)
    text = re.sub(r"\b\d{1,3}(?:gb|g|tb)\b", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# 토큰 단계 상품명 클러스터링 최종본입니다.
# 1) 기존 cluster_normalized_name() 규칙으로 거래 문구/관리번호를 제거합니다.
# 2) keyword별 토큰 사전을 만들고, 공백 없이 붙어 있는 상품명도 긴 토큰 우선으로 다시 토큰화합니다.
# 3) c1 -> c2 -> c3 순서로 이전 토큰 경로 안에서 반복 클러스터링합니다.
# 4) c1, c2, ...를 합친 cluster_product_name을 최종 상품명 후보로 사용합니다.
TOKEN_STAGE_DISTANCE_THRESHOLD = 0.25
TOKEN_STAGE_MIN_CLUSTER_ITEMS = 2
TOKEN_STAGE_MAX_COLUMNS = 8
TOKEN_STAGE_TREE_ITEM_LIMIT = 20
TOKEN_STAGE_REVIEW_KEYWORD = None  # 예: "갤럭시", "아이폰". 전체 조회는 None
TOKEN_STAGE_REVIEW_LIMIT = 300
TOKEN_STAGE_MIN_VOCAB_TOKEN_LENGTH = 2
TOKEN_STAGE_MIN_VOCAB_TOKEN_COUNT = 2
TOKEN_STAGE_TOKEN_NGRAM_RANGE = (1, 4)


def is_excluded_cluster_token(token):
    return str(token or "").strip() in TOKEN_STAGE_EXCLUDED_TOKENS


def split_space_tokens(value):
    return [
        token
        for token in str(value or "").split()
        if token and not is_excluded_cluster_token(token)
    ]


def build_keyword_token_vocabulary(names):
    token_counts = {}
    for name in names:
        for token in split_space_tokens(name):
            if len(token) < TOKEN_STAGE_MIN_VOCAB_TOKEN_LENGTH:
                continue
            if is_excluded_cluster_token(token):
                continue
            token_counts[token] = token_counts.get(token, 0) + 1

    vocabulary = [
        token
        for token, count in token_counts.items()
        if count >= TOKEN_STAGE_MIN_VOCAB_TOKEN_COUNT and not is_excluded_cluster_token(token)
    ]
    return sorted(vocabulary, key=lambda token: (-len(token), -token_counts[token], token))


def split_attached_token(token, token_vocabulary):
    if not token:
        return []

    matches = []
    occupied = [False] * len(token)
    for vocab_token in token_vocabulary:
        if vocab_token == token or len(vocab_token) >= len(token):
            continue

        start_idx = 0
        while True:
            found_idx = token.find(vocab_token, start_idx)
            if found_idx == -1:
                break

            end_idx = found_idx + len(vocab_token)
            if not any(occupied[found_idx:end_idx]):
                matches.append((found_idx, end_idx, vocab_token))
                for idx in range(found_idx, end_idx):
                    occupied[idx] = True
            start_idx = found_idx + 1

    if not matches:
        return [token]

    tokens = []
    cursor = 0
    for start_idx, end_idx, vocab_token in sorted(matches, key=lambda item: item[0]):
        if cursor < start_idx:
            tokens.append(token[cursor:start_idx])
        tokens.append(vocab_token)
        cursor = end_idx
    if cursor < len(token):
        tokens.append(token[cursor:])

    return [item for item in tokens if item]


def tokenize_cluster_name(value, token_vocabulary):
    tokens = []
    for token in split_space_tokens(value):
        tokens.extend(split_attached_token(token, token_vocabulary))
    return [token for token in tokens if token and not is_excluded_cluster_token(token)]


def choose_representative_token(tokens):
    token_counts = pd.Series(list(tokens)).value_counts()
    if token_counts.empty:
        return ""

    max_count = token_counts.iloc[0]
    candidates = token_counts[token_counts == max_count].index.tolist()
    return sorted(candidates, key=lambda token: (-len(token), token))[0]


def build_token_cluster_labels(tokens):
    tokens = pd.Series(tokens).fillna("").astype(str)
    labels = pd.Series("", index=tokens.index, dtype="object")
    non_empty_tokens = tokens[tokens != ""]
    unique_tokens = non_empty_tokens.drop_duplicates().tolist()

    if not unique_tokens:
        return labels
    if len(unique_tokens) < TOKEN_STAGE_MIN_CLUSTER_ITEMS:
        labels.loc[non_empty_tokens.index] = non_empty_tokens
        return labels

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=TOKEN_STAGE_TOKEN_NGRAM_RANGE,
        min_df=1,
    )
    token_vectors = vectorizer.fit_transform(unique_tokens)

    try:
        model = AgglomerativeClustering(
            n_clusters=None,
            metric="cosine",
            linkage="average",
            distance_threshold=TOKEN_STAGE_DISTANCE_THRESHOLD,
        )
    except TypeError:
        model = AgglomerativeClustering(
            n_clusters=None,
            affinity="cosine",
            linkage="average",
            distance_threshold=TOKEN_STAGE_DISTANCE_THRESHOLD,
        )

    token_cluster_df = pd.DataFrame(
        {
            "token": unique_tokens,
            "token_cluster_id": model.fit_predict(token_vectors.toarray()),
        }
    )
    token_frequency = non_empty_tokens.value_counts().to_dict()

    token_label_map = {}
    for _, cluster_df in token_cluster_df.groupby("token_cluster_id", sort=True):
        representative_token = choose_representative_token(
            token
            for token in cluster_df["token"]
            for _ in range(token_frequency.get(token, 0))
        )
        for token in cluster_df["token"]:
            token_label_map[token] = representative_token

    labels.loc[non_empty_tokens.index] = non_empty_tokens.map(token_label_map)
    return labels


def join_cluster_product_name(row, token_columns):
    tokens = [str(row[column]).strip() for column in token_columns if str(row[column]).strip()]
    return " ".join(tokens)


def add_token_stage_columns(keyword_df):
    keyword_df = keyword_df.copy().reset_index(drop=True)
    keyword_df["cluster_name_text"] = keyword_df["name"].apply(cluster_normalized_name)
    keyword_df.loc[keyword_df["cluster_name_text"] == "", "cluster_name_text"] = keyword_df[
        "name"
    ].astype(str)

    token_vocabulary = build_keyword_token_vocabulary(keyword_df["cluster_name_text"])
    keyword_df["_name_tokens"] = keyword_df["cluster_name_text"].apply(
        lambda value: tokenize_cluster_name(value, token_vocabulary)
    )
    keyword_df["tokenized_cluster_name_text"] = keyword_df["_name_tokens"].apply(" ".join)

    token_columns = []
    max_token_count = int(keyword_df["_name_tokens"].apply(len).max()) if len(keyword_df) else 0
    for token_position in range(min(max_token_count, TOKEN_STAGE_MAX_COLUMNS)):
        token_column = f"c{token_position + 1}"
        token_columns.append(token_column)
        keyword_df[token_column] = ""

        group_columns = token_columns[:-1] or ["keyword"]
        for _, stage_df in keyword_df.groupby(group_columns, sort=True, dropna=False):
            stage_tokens = stage_df["_name_tokens"].apply(
                lambda tokens: tokens[token_position] if token_position < len(tokens) else ""
            )
            keyword_df.loc[stage_df.index, token_column] = build_token_cluster_labels(stage_tokens).values

    keyword_df["cluster_product_name"] = keyword_df.apply(
        join_cluster_product_name,
        axis=1,
        token_columns=token_columns,
    )
    return keyword_df.drop(columns=["_name_tokens"]), token_columns


def summarize_token_stage_clusters(item_df, token_columns):
    if item_df.empty:
        return pd.DataFrame()

    summary_rows = []
    group_columns = ["keyword", *token_columns, "cluster_product_name"]
    for group_key, cluster_df in item_df.groupby(group_columns, sort=True, dropna=False):
        group_values = dict(zip(group_columns, group_key if isinstance(group_key, tuple) else (group_key,)))
        price_median = cluster_df["price_numeric"].median()
        summary_rows.append(
            {
                **group_values,
                "item_count": len(cluster_df),
                "min_price": int(cluster_df["price_numeric"].min()),
                "median_price": float(price_median),
                "average_price": float(cluster_df["price_numeric"].mean()),
                "max_price": int(cluster_df["price_numeric"].max()),
                "sample_tokenized_names": " | ".join(
                    dict.fromkeys(cluster_df["tokenized_cluster_name_text"].astype(str).head(5))
                ),
                "sample_normalized_names": " | ".join(
                    dict.fromkeys(cluster_df["cluster_name_text"].astype(str).head(5))
                ),
                "sample_original_names": " | ".join(
                    dict.fromkeys(cluster_df["name"].astype(str).head(5))
                ),
            }
        )

    summary_df = pd.DataFrame(summary_rows).sort_values(
        ["keyword", "item_count", "median_price"], ascending=[True, False, True]
    )

    front_columns = ["keyword", *token_columns, "cluster_product_name"]
    return summary_df[front_columns + [column for column in summary_df.columns if column not in front_columns]]


def build_token_stage_cluster_tree(item_df, summary_df, token_columns):
    tree_rows = []
    if item_df.empty or summary_df.empty:
        return tree_rows

    for keyword, keyword_summary_df in summary_df.groupby("keyword", sort=True):
        keyword_item_df = item_df[item_df["keyword"] == keyword]
        clusters = []
        for _, summary_row in keyword_summary_df.iterrows():
            mask = keyword_item_df["cluster_product_name"] == summary_row["cluster_product_name"]
            for token_column in token_columns:
                mask &= keyword_item_df[token_column] == summary_row[token_column]
            cluster_item_df = keyword_item_df[mask]
            clusters.append(
                {
                    "cluster_product_name": summary_row["cluster_product_name"],
                    "token_path": {column: summary_row[column] for column in token_columns if summary_row[column]},
                    "item_count": int(summary_row["item_count"]),
                    "price_summary": {
                        "min": int(summary_row["min_price"]),
                        "median": float(summary_row["median_price"]),
                        "average": float(summary_row["average_price"]),
                        "max": int(summary_row["max_price"]),
                    },
                    "items": cluster_item_df[
                        ["platform", "pid", "name", "price_numeric", "status", "link"]
                    ].head(TOKEN_STAGE_TREE_ITEM_LIMIT).to_dict("records"),
                }
            )

        tree_rows.append(
            {
                "keyword": keyword,
                "item_count": len(keyword_item_df),
                "cluster_count": len(keyword_summary_df),
                "clusters": sorted(clusters, key=lambda item: item["item_count"], reverse=True),
            }
        )

    return tree_rows


def build_keyword_token_stage_clusters(source_df):
    item_frames = []
    token_column_count = 0

    for _, keyword_df in source_df.groupby("keyword", sort=True):
        keyword_item_df, keyword_token_columns = add_token_stage_columns(keyword_df)
        item_frames.append(keyword_item_df)
        token_column_count = max(token_column_count, len(keyword_token_columns))

    item_df = pd.concat(item_frames, ignore_index=True) if item_frames else pd.DataFrame()
    token_columns = [f"c{idx}" for idx in range(1, token_column_count + 1)]
    for token_column in token_columns:
        if token_column not in item_df.columns:
            item_df[token_column] = ""
    item_df[token_columns] = item_df[token_columns].fillna("")

    summary_df = summarize_token_stage_clusters(item_df, token_columns)
    tree_rows = build_token_stage_cluster_tree(item_df, summary_df, token_columns)
    return item_df, summary_df, tree_rows


token_stage_cluster_item_df, token_stage_cluster_summary_df, token_stage_cluster_tree = (
    build_keyword_token_stage_clusters(clean_price_df)
)

token_stage_review_df = token_stage_cluster_summary_df.copy()
if TOKEN_STAGE_REVIEW_KEYWORD:
    token_stage_review_df = token_stage_review_df[
        token_stage_review_df["keyword"].astype(str).str.contains(
            TOKEN_STAGE_REVIEW_KEYWORD, case=False, na=False
        )
    ]

TOKEN_STAGE_REVIEW_CSV_PATH = Path("exports") / "token_stage_cluster_review_df.csv"
TOKEN_STAGE_REVIEW_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
token_stage_review_df.to_csv(TOKEN_STAGE_REVIEW_CSV_PATH, index=False, encoding="utf-8-sig")

print(
    f"keyword 수: {len(token_stage_cluster_tree):,}, "
    f"token-stage cluster 수: {len(token_stage_cluster_summary_df):,}, "
    f"대상 상품 수: {len(token_stage_cluster_item_df):,}, "
    f"CSV 저장: {TOKEN_STAGE_REVIEW_CSV_PATH}"
)
token_stage_review_df.head(TOKEN_STAGE_REVIEW_LIMIT)

keyword 수: 2, token-stage cluster 수: 8,310, 대상 상품 수: 14,245, CSV 저장: exports\token_stage_cluster_review_df.csv


,keyword,c1,c2,c3,c4,c5,c6,c7,c8,cluster_product_name,item_count,min_price,median_price,average_price,max_price,sample_tokenized_names,sample_normalized_names,sample_original_names
1688,갤럭시,갤럭시a16,라이트그린,무,잔상,,,,,갤럭시a16 라이트그린 무 잔상,145,165000,165000.0,165000.00000,165000,갤럭시a16 라이트그린 무 잔상,갤럭시a16라이트그린 무잔상,"갤럭시A16라이트그린,무잔상(3337)"
1692,갤럭시,갤럭시a33,블랙,,,,,,,갤럭시a33 블랙,143,127000,132000.0,131580.41958,132000,갤럭시a33 블랙,갤럭시a33블랙,갤럭시A33블랙(6380)
2326,갤럭시,갤럭시버디4,블랙,a,플러스,급,무,잔상,,갤럭시버디4 블랙 a 플러스 급 무 잔상,132,165000,165000.0,165000.00000,165000,갤럭시버디4 블랙 a 플러스 급 무 잔상,갤럭시버디4블랙a 플러스 급 무잔상,"갤럭시버디4블랙A+급,무잔상(5139)"
2050,갤럭시,갤럭시s22,핑크,무잔상,,,,,,갤럭시s22 핑크 무잔상,131,245000,245000.0,245000.00000,245000,갤럭시s22 핑크 무잔상,갤럭시s22핑크무잔상,갤럭시S22핑크무잔상(3663)
2321,갤럭시,갤럭시노트8,블루,무,잔상,께끗,,,,갤럭시노트8 블루 무 잔상 께끗,130,105000,105000.0,105000.00000,105000,갤럭시노트8 블루 무 잔상 께끗,갤럭시노트8블루 무잔상 께끗,"갤럭시노트8블루,무잔상,께끗(1925)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
945,갤럭시,갤럭시,노트20,256,기가,그레이,,,,갤럭시 노트20 256 기가 그레이,4,155000,167500.0,167500.00000,180000,갤럭시 노트20 256 기가 그레이,갤럭시노트20 256기가 그레이,갤럭시노트20 256기가 그레이 / 0428 | 갤럭시노트20 256기가 그레이 /...
954,갤럭시,갤럭시,노트20,256,기가,미스틱,블루,부산,폰,갤럭시 노트20 256 기가 미스틱 블루 부산 폰,4,171000,171000.0,171000.00000,171000,갤럭시 노트20 256 기가 미스틱 블루 부산 폰 0 19 34 서울 춘천 마산,갤럭시노트20 256기가 미스틱블루 부산 폰 01934 서울 춘천 마산,갤럭시노트20 256기가 미스틱블루 부산중고폰 01934 서울 춘천 마산
719,갤럭시,갤럭시,s22,플러스,핑크,무,잔상,정상,공기기,갤럭시 s22 플러스 핑크 무 잔상 정상 공기기,4,180000,180000.0,180000.00000,180000,갤럭시 s22 플러스 핑크 무 잔상 정상 공기기 폰,갤럭시 s22플러스 핑크 무잔상 정상공기기 폰,갤럭시 S22플러스 핑크 무잔상 정상공기기 중고폰
1804,갤럭시,갤럭시s2,1,256,기가,팬텀,그레이,부산,폰,갤럭시s2 1 256 기가 팬텀 그레이 부산 폰,4,182000,182000.0,182000.00000,182000,갤럭시s2 1 256 기가 팬텀 그레이 부산 폰 0 24 16 하동 대전 창원,갤럭시s21 256기가 팬텀그레이 부산 폰 02416 하동 대전 창원,갤럭시S21 256기가 팬텀그레이 부산중고폰 02416 하동 대전 창원


In [ ]:
# DB 저장용 item-level 결과 확인용 DF입니다.
# name은 원본 상품명 그대로 보존하고, cluster_product_name을 매칭된 상품명 후보로 사용합니다.
DB_SAVE_PREVIEW_LIMIT = 300
DB_SAVE_CSV_PATH = Path("exports") / "keyword_db_save_preview_df.csv"

cluster_token_columns = [column for column in token_stage_cluster_item_df.columns if re.fullmatch(r"c\d+", column)]
base_db_columns = [
    "keyword",
    "platform",
    "pid",
    "name",
    "cluster_product_name",
    "price_numeric",
    "status",
    "link",
    "cluster_name_text",
    "tokenized_cluster_name_text",
]
db_save_columns = [column for column in [*base_db_columns, *cluster_token_columns] if column in token_stage_cluster_item_df.columns]

db_save_df = token_stage_cluster_item_df[db_save_columns].copy()
db_save_df = db_save_df.sort_values(
    ["keyword", "cluster_product_name", "price_numeric", "platform", "pid"],
    ascending=[True, True, True, True, True],
).reset_index(drop=True)

DB_SAVE_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
db_save_df.to_csv(DB_SAVE_CSV_PATH, index=False, encoding="utf-8-sig")

print(
    f"DB 저장용 row 수: {len(db_save_df):,}, "
    f"매칭 상품명 후보 수: {db_save_df['cluster_product_name'].nunique():,}, "
    f"CSV 저장: {DB_SAVE_CSV_PATH}"
)

db_save_df.head(DB_SAVE_PREVIEW_LIMIT)

DB 저장용 row 수: 14,245, 매칭 상품명 후보 수: 8,307, CSV 저장: exports\keyword_db_save_preview_df.csv


,keyword,platform,pid,name,cluster_product_name,price_numeric,status,link,cluster_name_text,tokenized_cluster_name_text,c1,c2,c3,c4,c5,c6,c7,c8
0,갤럭시,번개장터,412344703,100~105갤럭시 하계 자켓,10 0 105 갤럭시 하계 자켓,45000,판매중,https://m.bunjang.co.kr/products/412344703,100 105갤럭시 하계 자켓,10 0 105 갤럭시 하계 자켓,10,0,105,갤럭시,하계,자켓,,
1,갤럭시,번개장터,371415413,10.5 / 루이비통 갤럭시 모노그램 스니커즈,10 5 루이비통 갤럭시 모노그램 스니커즈,600000,판매중,https://m.bunjang.co.kr/products/371415413,10 5 루이비통 갤럭시 모노그램 스니커즈,10 5 루이비통 갤럭시 모노그램 스니커즈,10,5,루이비통,갤럭시,모노그램,스니커즈,,
2,갤럭시,번개장터,400954249,100사이즈 갤럭시 캐시미어 혼방 숏 코트 판매합니다,100 사이즈 갤럭시 캐시미어 혼방 숏 코트,45000,판매중,https://m.bunjang.co.kr/products/400954249,100사이즈 갤럭시 캐시미어 혼방 숏 코트,100 사이즈 갤럭시 캐시미어 혼방 숏 코트,100,사이즈,갤럭시,캐시미어,혼방,숏,코트,
3,갤럭시,번개장터,410334184,(13만) 갤럭시 점프2 KT 화이트 128GB 직거래/퀵/택배,13 만 갤럭시 점프 2 kt 화이트 퀵,148000,판매중,https://m.bunjang.co.kr/products/410334184,13만 갤럭시 점프2 kt 화이트 퀵 택배,13 만 갤럭시 점프 2 kt 화이트 퀵 택배,13,만,갤럭시,점프,2,kt,화이트,퀵
4,갤럭시,번개장터,402652991,19.3.30) S10e 갤럭시 삼성 공기계 중고 휴대폰 파라요~,19 3 30 s10 e 갤럭시 삼성 공기,130000,판매중,https://m.bunjang.co.kr/products/402652991,19 3 30 s10e 갤럭시 삼성 공기계 휴대폰 파라요,19 3 30 s10 e 갤럭시 삼성 공기 계 휴대폰 파라요,19,3,30,s10,e,갤럭시,삼성,공기
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,갤럭시,번개장터,402046897,Freewell 프리웰 갤럭시 s25 울트라 프로 필름메이커 케이스,freewell 프리웰 갤럭시 s 25 울트라 프로 필름,50000,판매중,https://m.bunjang.co.kr/products/402046897,freewell 프리웰 갤럭시 s25 울트라 프로 필름메이커 케이스,freewell 프리웰 갤럭시 s 25 울트라 프로 필름 메이커 케이스,freewell,프리웰,갤럭시,s,25,울트라,프로,필름
296,갤럭시,번개장터,408145737,galaxxxy (갤럭시) BUCKET HOMIES 하라주쿠 스카잔,galax xxy 갤럭시 bucket homies 하라주쿠 스카잔,55000,판매중,https://m.bunjang.co.kr/products/408145737,galaxxxy 갤럭시 bucket homies 하라주쿠 스카잔,galax xxy 갤럭시 bucket homies 하라주쿠 스카잔,galax,xxy,갤럭시,bucket,homies,하라주쿠,스카잔,
297,갤럭시,번개장터,409958888,Galaxy Tab 갤럭시탭 S9FE 128GB Wi-Fi,galax y tab 갤럭시 탭 s9 fe wi,350000,판매중,https://m.bunjang.co.kr/products/409958888,galaxy tab 갤럭시탭 s9fe wi fi,galax y tab 갤럭시 탭 s9 fe wi fi,galax,y,tab,갤럭시,탭,s9,fe,wi
298,갤럭시,번개장터,398674820,Galaxy Z Flip 5 512GB 팔아요 (그레이),galax y z flip 5 그레이,300000,판매중,https://m.bunjang.co.kr/products/398674820,galaxy z flip 5 그레이,galax y z flip 5 그레이,galax,y,z,flip,5,그레이,,
